In [ ]:
import torch
from protlig.testing_helpers import ( 
    GCNLayerSparse, load_dataset, mol_to_graph, 
    iterate_minibatches, global_add_pool, collate)

import torch.nn as nn


In [2]:
class GCNConv(nn.Module):
    def __init__(self, in_dims, hidden_dims, n_layers):
        super().__init__()
        dims = [in_dims] + [hidden_dims] * n_layers

        self.convs = nn.ModuleList(
            GCNLayerSparse(dims[i], dims[i + 1]) for i in range(n_layers))

        self.read_out = nn.Sequential(
            nn.Linear(hidden_dims, hidden_dims),
            nn.ReLU(),
            nn.Linear(hidden_dims, 1)
        )

    def forward(self, x, edge_index, batch, n_graphs):
        h = x
        for conv in self.convs:
            h = torch.relu(conv(h, edge_index))
        g = global_add_pool(h, batch, n_graphs)
        return self.read_out(g).squeeze(-1)

In [3]:
def train_full_batch(data, model, epochs=3000, lr=1e-2, log_every=250):
    x, ei, batch, y = data
    n_graphs = int(max(batch)) + 1
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for e in range(1, epochs + 1):
        opt.zero_grad()
        out = model(x, ei, batch, n_graphs)
        loss = ((y - out) ** 2).mean()
        loss.backward()
        opt.step()

        history.append(loss.detach().item())
        if e % log_every == 0 or e == 1:
            print(f"  epoch {e:4d}   mse {history[-1]:.4e}   "
                  f"rmse {history[-1] ** 0.5:.4e}")
    return history



In [4]:
data = load_dataset()
smiles, labels = data['SMILES'].values, data['label'].values
graphs = [mol_to_graph(s, l) for s, l in zip(smiles, labels)]
collated = collate(graphs)

IN_DIMS = graphs[0][0].size(1)
HIDDEN_DIMS = 32
N_LAYERS = 3

model = GCNConv(IN_DIMS, HIDDEN_DIMS, N_LAYERS)

history = train_full_batch(collated, model)

/Users/tungvuduc/projects/protlig/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  epoch    1   mse 1.7018e+01   rmse 4.1253e+00
  epoch  250   mse 6.6793e-01   rmse 8.1727e-01
  epoch  500   mse 4.5595e-01   rmse 6.7524e-01
  epoch  750   mse 4.0501e-01   rmse 6.3640e-01
  epoch 1000   mse 3.7187e-01   rmse 6.0981e-01
  epoch 1250   mse 3.0903e-01   rmse 5.5590e-01
  epoch 1500   mse 2.7771e-01   rmse 5.2698e-01
  epoch 1750   mse 2.5426e-01   rmse 5.0424e-01
  epoch 2000   mse 2.4559e-01   rmse 4.9557e-01
  epoch 2250   mse 2.2638e-01   rmse 4.7579e-01
  epoch 2500   mse 2.3560e-01   rmse 4.8539e-01
  epoch 2750   mse 2.3166e-01   rmse 4.8131e-01
  epoch 3000   mse 2.6467e-01   rmse 5.1446e-01
